# DeepEval metric lab: follow the inputs, prompts, and arithmetic

**Fully offline • fictional data • authored judge responses • DeepEval 4.3.2**

Run every cell without credentials or network requests. The real installed metric classes assemble prompts, validate response schemas, and calculate scores; `TeachingJudge` supplies responses we wrote in advance. This demonstrates **how the machinery works**, not the accuracy of an AI judge.

You will reproduce faithfulness **2/3**, rubric coverage **0.8**, and question-based summarization coverage **2/3**. Removing an important assessment question will expose a coverage blind spot.

## Goal: distinguish three meanings of “question”

| Item | What it means | Must you provide it? |
|---|---|---|
| `LLMTestCase.input` | A metric-specific input field; our evaluator puts policy/instructions here. | Required by the metrics used below; it need not be a literal question. |
| `assessment_questions` | Evaluation probes used by `SummarizationMetric`, usually designed for a “yes” answer from the source. | Optional: DeepEval generates them when omitted; its default `n` is 5. |
| Meeting Questions and Answers | Questions participants actually asked, and their answers or explicit absence of an answer. | Summarize only what occurred. Generated assessment questions are not meeting Q&A. |

No `expected_output` or human-written reference summary is required by these configured metrics. That does **not** remove the need for human calibration. [Summarization documentation](https://deepeval.com/docs/metrics-summarization), [Faithfulness documentation](https://deepeval.com/docs/metrics-faithfulness), [G-Eval documentation](https://deepeval.com/docs/metrics-llm-evals).

## Setup

Install the repository's `.[dev,notebooks]` extras and select that environment as the notebook kernel. This lab never loads `.env`, constructs `ChatOpenAI`, or calls a hosted evaluation service. Import the package before DeepEval so its privacy settings are applied.

Field meanings depend on the metric:

| Field | Our faithfulness / coverage configuration | DeepEval `SummarizationMetric` |
|---|---|---|
| `input` | Generic evaluation policy and optional audience/purpose context | The full source text to summarize |
| `actual_output` | Candidate summary | Candidate summary |
| `retrieval_context` | Full transcript and optional approved context, as labeled records | Not used for the source in this lab |
| `expected_output` | Not used | Not used |

The name `retrieval_context` comes from DeepEval; this repository supplies a complete source rather than running a retrieval system.

In [ ]:
# Check the active kernel before importing the evaluator or its dependencies.
import sys
import tomllib
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file() and (path / "src/meeting_eval").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the meeting-summary-eval repository or its notebooks folder.")
dependencies = tomllib.loads((ROOT / "pyproject.toml").read_text(encoding="utf-8"))["project"]["dependencies"]
required_deepeval = next(item.removeprefix("deepeval==") for item in dependencies if item.startswith("deepeval=="))
try:
    installed_deepeval = version("deepeval")
except PackageNotFoundError:
    installed_deepeval = "not installed"
print("Python:", sys.executable)
print("DeepEval:", installed_deepeval, "| Required:", required_deepeval)
if installed_deepeval != required_deepeval:
    raise RuntimeError(
        f"This kernel needs deepeval=={required_deepeval}; found {installed_deepeval}. "
        "From the repository root, run %pip install --upgrade -e '.[dev,notebooks]' "
        "to install into the active kernel, then restart the kernel and rerun from the top."
    )


In [ ]:
from pathlib import Path
from importlib.metadata import version
import ast
import inspect
import json
import math

import meeting_eval  # Applies the package's local/offline-friendly DeepEval settings.
from meeting_eval.policy import POLICY
from meeting_eval.metrics import COVERAGE_STEPS, COVERAGE_RUBRIC, create_metrics
from deepeval.metrics import SummarizationMetric
from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase
from IPython.display import Markdown, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists())
print('DeepEval:', version('deepeval'))
print('Mode: OFFLINE — authored responses only')

def show_table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines += ['| ' + ' | '.join(str(value).replace('|', r'\|').replace('\n', ' ') for value in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))

## 1. Read the fictional source and three small candidate excerpts

The excerpts isolate metric behavior. Their layout is accepted as supplied; no required headings or sections are used. The faithfulness excerpt contains a deliberate certainty error; the question-coverage excerpt is faithful but omits important information.

In [ ]:
TRANSCRIPT = """FICTIONAL SOFTWARE TEACHING FIXTURE
Alex: Short-term yields fell 10 basis points this week.
Alex: Longer-term yields were unchanged.
Jordan: Inflation is moderating unevenly. One reading does not establish a durable trend.
Jordan: No date for a rate cut was forecast.
Sam: A higher yield does not eliminate price risk.
Casey: Is a rate cut certain next month?
Jordan: No. One reading is insufficient.
Sam: The same caution applies when explaining this week's figures to advisors.
"""
FAITHFULNESS_SUMMARY = (
    'Short-term yields fell 10 basis points this week. '
    'Longer-term yields were unchanged. A rate cut is certain next month.'
)
COVERAGE_SUMMARY = (
    'Short-term yields fell 10 basis points this week while longer-term yields were unchanged. '
    'Inflation is moderating unevenly; one reading does not establish a durable trend. '
    'No rate-cut date was forecast and next month is not certain. '
    'A higher yield does not eliminate price risk.'
)
QUESTION_SUMMARY = (
    'Short-term yields fell 10 basis points this week. '
    'Longer-term yields were unchanged. No date for a rate cut was forecast.'
)
print(TRANSCRIPT)
show_table(['Candidate', 'Text'], [
    ['Faithfulness example', FAITHFULNESS_SUMMARY],
    ['Holistic coverage example', COVERAGE_SUMMARY],
    ['Question coverage example', QUESTION_SUMMARY],
])

### A schema-aware teaching judge

A normal judge reads each prompt and generates a judgment. This one only returns authored fixtures in the exact schemas requested by DeepEval. It also captures the real prompts. There is no language-model reasoning here: **editing the transcript alone will not make its responses adapt**.

The class uses `schema.__module__` as well as the schema name because faithfulness and summarization both have a class named `Verdicts`. For question-answer calls it reads the actual question list from the installed template and looks up authored answers.

In [ ]:
SOURCE_TRUTHS = [
    '[transcript] Short-term yields fell 10 basis points this week.',
    '[transcript] Longer-term yields were unchanged.',
    '[transcript] Jordan said inflation is moderating unevenly; one reading does not establish a durable trend.',
    '[transcript] No date for a rate cut was forecast; a cut next month is not certain.',
    '[transcript] A higher yield does not eliminate price risk.',
]
FAITHFULNESS_CLAIMS = [
    'Short-term yields fell 10 basis points this week.',
    'Longer-term yields were unchanged.',
    'A rate cut is certain next month.',
]
QUESTION_CLAIMS = [
    'Short-term yields fell 10 basis points this week.',
    'Longer-term yields were unchanged.',
    'No date for a rate cut was forecast.',
]
Q_SHORT = 'Did short-term yields fall 10 basis points this week?'
Q_LONG = 'Were longer-term yields unchanged?'
Q_RISK = 'Did the meeting state that higher yield does not eliminate price risk?'
Q_INFLATION = 'Did the meeting qualify inflation moderation as uneven?'
Q_NO_DATE = 'Did the meeting avoid forecasting a rate-cut date?'
Q_NOT_IN_SOURCE = 'Did the meeting launch a new named fund?'
FIXED_QUESTIONS = [Q_SHORT, Q_LONG, Q_RISK]
GENERATED_QUESTIONS = [Q_SHORT, Q_LONG, Q_RISK, Q_INFLATION, Q_NO_DATE]
# Authored judgments: value = (answer for full source, answer for QUESTION_SUMMARY).
AUTHORED_ANSWERS = {
    Q_SHORT: ('yes', 'yes'), Q_LONG: ('yes', 'yes'), Q_RISK: ('yes', 'no'),
    Q_INFLATION: ('yes', 'no'), Q_NO_DATE: ('yes', 'yes'),
    Q_NOT_IN_SOURCE: ('no', 'no'),
}

class TeachingJudge(DeepEvalBaseLLM):
    def __init__(self, scenario):
        self.scenario = scenario
        self.calls = []

    def load_model(self):
        return self

    def get_model_name(self):
        return 'offline-authored-teaching-judge'

    def generate(self, prompt, schema=None):
        if schema is None:
            raise ValueError('This lab expects schema-based calls only.')
        self.calls.append({'schema': f'{schema.__module__}.{schema.__name__}', 'prompt': prompt})
        name = schema.__name__
        if name == 'Truths':
            data = {'truths': SOURCE_TRUTHS}
        elif name == 'Claims':
            data = {'claims': FAITHFULNESS_CLAIMS if self.scenario == 'faithfulness' else QUESTION_CLAIMS}
        elif name == 'Verdicts':
            verdicts = [{'verdict': 'yes', 'reason': None} for _ in range(3)]
            if self.scenario == 'faithfulness':
                verdicts[-1] = {'verdict': 'no', 'reason': 'The source explicitly says a cut next month is not certain.'}
            data = {'verdicts': verdicts}
        elif name == 'FaithfulnessScoreReason':
            data = {'reason': 'Two extracted claims are supported; certainty about next month contradicts the source.'}
        elif name == 'ReasonScore':
            data = {'score': 8, 'reason': 'The central developments and caveats are retained; a modest advisor-explanation detail is omitted.'}
        elif name == 'Questions':
            # SummarizationMetric asks for its default n=5 when no list is supplied.
            assert 'generate 5 closed-ended questions' in prompt
            data = {'questions': GENERATED_QUESTIONS}
        elif name == 'Answers':
            text_and_questions = prompt.rsplit('\nText:\n', 1)[1]
            text, question_section = text_and_questions.split('\nQuestions:\n', 1)
            questions = ast.literal_eval(question_section.rsplit('\nJSON:', 1)[0].strip())
            if text.strip() == TRANSCRIPT.strip():
                answer_index = 0
            elif text.strip() == QUESTION_SUMMARY.strip():
                answer_index = 1
            else:
                raise ValueError('Changed fixture: author explicit teaching answers for the new text.')
            data = {'answers': [AUTHORED_ANSWERS[question][answer_index] for question in questions]}
        elif name == 'SummarizationScoreReason':
            data = {'reason': 'The scripted alignment verdicts support all claims; question coverage depends on which source-supported probes are retained.'}
        else:
            raise ValueError(f'Unscripted schema: {schema.__module__}.{name}')
        return schema.model_validate(data)

    async def a_generate(self, prompt, schema=None):
        return self.generate(prompt, schema)

## 2. Run the repository's real faithfulness metric

`create_metrics` configures the installed `FaithfulnessMetric` with our meeting-specific templates, ambiguous claims penalized, and `threshold=None` for score-only operation. It extracts source statements and summary claims, then asks for one verdict per claim and a reason.

The full transcript is preserved in `retrieval_context`. The verdict stage itself uses the **extracted** source inventory; extraction can lose information. The production evaluator also rejects incomplete/empty extraction and mismatched verdict counts.

In [ ]:
faithfulness_judge = TeachingJudge('faithfulness')
faithfulness = create_metrics(faithfulness_judge, ('faithfulness',))['faithfulness']
faithfulness_case = LLMTestCase(
    input=POLICY,
    actual_output=FAITHFULNESS_SUMMARY,
    retrieval_context=[json.dumps({'source': 'transcript', 'text': TRANSCRIPT})],
)
await faithfulness.a_measure(faithfulness_case, _show_indicator=False)
show_table(['Extracted claim', 'Verdict', 'Reason'], [
    [claim, verdict.verdict, verdict.reason or 'Source-supported']
    for claim, verdict in zip(faithfulness.claims, faithfulness.verdicts)
])
supported = sum(verdict.verdict == 'yes' for verdict in faithfulness.verdicts)
print(f'Manual score: {supported}/{len(faithfulness.claims)} = {supported / len(faithfulness.claims):.3f}')
print(f'DeepEval score: {faithfulness.score:.3f}')
print('Extracted source statements:', len(faithfulness.truths))
print('Reason:', faithfulness.reason)
assert math.isclose(faithfulness.score, 2 / 3)
assert faithfulness_case.expected_output is None

### Inspect the source inventory and actual prompts

A source inventory is generated text, not independently verified truth. Attribution and caveats belong in that inventory. Here it is authored for teaching. The request log below is produced by the real metric, not a diagram of an imagined call sequence.

In [ ]:
show_table(['Authored extracted source statements'], [[truth] for truth in faithfulness.truths])
show_table(['Call', 'Requested response schema', 'Prompt characters'], [
    [index, call['schema'].rsplit('.', 1)[-1], len(call['prompt'])]
    for index, call in enumerate(faithfulness_judge.calls, start=1)
])
# Keep previews bounded; the complete captured prompt remains available in this variable.
verdict_prompt = next(call['prompt'] for call in faithfulness_judge.calls if call['schema'].endswith('.Verdicts'))
print(verdict_prompt[:650] + '\n... [middle omitted from preview] ...\n' + verdict_prompt[-800:])

## 3. Run holistic coverage with the real G-Eval configuration

This is a **different candidate excerpt** containing the major source meaning and qualifications. Our authored judge emits raw score 8; the configured 0–10 rubric normalizes it to 0.8. This is not “80% of transcript sentences.” The rubric can prioritize consequential omissions instead of giving every probe equal weight.

Explicit `evaluation_steps` avoid generating new steps on each run. This custom teaching judge supplies no token log probabilities, so the result uses the emitted rubric score rather than log-probability weighting. [G-Eval documentation](https://deepeval.com/docs/metrics-llm-evals).

In [ ]:
coverage_judge = TeachingJudge('coverage')
coverage = create_metrics(coverage_judge, ('coverage',))['coverage']
coverage_case = LLMTestCase(
    input=POLICY,
    actual_output=COVERAGE_SUMMARY,
    retrieval_context=[json.dumps({'source': 'transcript', 'text': TRANSCRIPT})],
)
await coverage.a_measure(coverage_case, _show_indicator=False)
show_table(['Raw rubric band', 'Meaning'], [
    [f'{rubric.score_range[0]}–{rubric.score_range[1]}', rubric.expected_outcome]
    for rubric in COVERAGE_RUBRIC
])
print(f'Authored emitted score: 8/10 → DeepEval score: {coverage.score:.1f}')
print('Fixed evaluation steps:', len(coverage.evaluation_steps))
print('Judge requests:', len(coverage_judge.calls))
print('Reason:', coverage.reason)
assert coverage.score == 0.8
assert len(coverage_judge.calls) == 1
assert coverage_case.expected_output is None

### Read the exact policy instead of guessing from the metric name

The files below are the full editable sources; the preview only shows the ends of the rendered G-Eval request. `coverage_judge.calls[0]["prompt"]` holds the full rendered request in memory. Reading that string makes no model call.

- [Metric templates, steps, and rubrics](../src/meeting_eval/metrics.py)
- [Shared evaluation policy](../src/meeting_eval/policy.py)
- [Full evaluator, including validation and diagnostics](../src/meeting_eval/evaluator.py)

In [ ]:
source_paths = [
    ROOT / 'src/meeting_eval/metrics.py',
    ROOT / 'src/meeting_eval/policy.py',
]
for path in source_paths:
    assert path.is_file()
    print(path.resolve())
coverage_prompt = coverage_judge.calls[0]['prompt']
print('\nRendered coverage prompt preview:\n')
print(coverage_prompt[:600] + '\n... [middle omitted from preview] ...\n' + coverage_prompt[-950:])

## 4. Try question-based coverage with `SummarizationMetric`

This comparison metric is **not added to the production defaults**. Its source belongs in `input`, and you can supply `assessment_questions` or let DeepEval generate them. We supply three fixed questions first. The fictional meeting's participant question is separate from these evaluation probes.

Our question-coverage candidate includes the yield moves and no-date statement, but omits the price-risk warning. All three extracted claims align with the source. Inspect the actual constructor and installed source path below; the lab follows version 4.3.2. [Summarization documentation](https://deepeval.com/docs/metrics-summarization).

In [ ]:
print(inspect.signature(SummarizationMetric))
print('Installed implementation:', Path(inspect.getfile(SummarizationMetric)).resolve())
summary_case = LLMTestCase(input=TRANSCRIPT, actual_output=QUESTION_SUMMARY)

async def run_question_metric(questions=None):
    judge = TeachingJudge('summarization')
    metric = SummarizationMetric(
        model=judge, threshold=None, eval_mode='llm', include_reason=True,
        assessment_questions=questions,
    )
    await metric.a_measure(summary_case, _show_indicator=False)
    return metric, judge

fixed_metric, fixed_judge = await run_question_metric(FIXED_QUESTIONS)
show_table(['Assessment question', 'Source answer', 'Summary answer'], [
    [verdict.question, verdict.original_verdict, verdict.summary_verdict]
    for verdict in fixed_metric.coverage_verdicts
])
print('Score breakdown:', fixed_metric.score_breakdown)
print('Overall score:', round(fixed_metric.score, 3))
assert summary_case.retrieval_context is None
assert summary_case.expected_output is None

### Recompute the denominator

Coverage counts probes with a **source “yes”** in its denominator, and counts those with both source and summary “yes” in its numerator. Alignment and coverage are different quantities; the overall summarization score is their minimum in this configuration.

A source answer of “no” excludes a question from the denominator; it should not be mistaken for evidence that the topic was covered. The answer prompt asks whether the text contains sufficient information for each probe: “no” can mean insufficient evidence, not proof that a factual proposition is false. Unknown or ambiguous source meaning needs inspection. The built-in answer prompt asks for binary yes/no, not a rich abstention verdict.

In [ ]:
eligible = [verdict for verdict in fixed_metric.coverage_verdicts
            if verdict.original_verdict.strip().lower() == 'yes']
retained = [verdict for verdict in eligible if verdict.summary_verdict.strip().lower() == 'yes']
manual_coverage = len(retained) / len(eligible)
alignment = fixed_metric.score_breakdown['Alignment']
print(f'Question coverage = {len(retained)}/{len(eligible)} = {manual_coverage:.3f}')
print(f'Alignment = {alignment:.3f}')
print(f'Overall = min({alignment:.3f}, {manual_coverage:.3f}) = {fixed_metric.score:.3f}')
assert math.isclose(manual_coverage, 2 / 3)
assert math.isclose(fixed_metric.score_breakdown['Coverage'], manual_coverage)
assert fixed_metric.score == min(fixed_metric.score_breakdown.values())

## 5. Omit the question list: DeepEval generates it internally

The next constructor receives no assessment-question list. The real metric makes a `Questions` schema request using its default `n=5`, then asks for source and summary answers. Our authored `Questions` response supplies five probes so this is still offline. A live judge would generate the list, making question selection another source of variation.

In [ ]:
generated_metric, generated_judge = await run_question_metric()
question_calls = [call for call in generated_judge.calls if call['schema'].endswith('.Questions')]
print('Default n:', generated_metric.n)
print('Question-generation requests:', len(question_calls))
show_table(['Generated-path assessment question', 'Source', 'Summary'], [
    [verdict.question, verdict.original_verdict, verdict.summary_verdict]
    for verdict in generated_metric.coverage_verdicts
])
print('Score breakdown:', generated_metric.score_breakdown)
assert generated_metric.n == 5 and len(question_calls) == 1
assert len(generated_metric.assessment_questions) == 5
assert math.isclose(generated_metric.score_breakdown['Coverage'], 3 / 5)

## 6. Exercise: make an omission disappear by removing its probe

Keep the source and candidate unchanged. Predict the result before running:

1. Remove the price-risk question: the two yield questions cannot detect that omission.
2. Restore it: the omission becomes measurable.
3. Add a fund-launch question the source cannot answer “yes”: it does not enlarge the source-yes denominator.

These are authored response fixtures. The exercise isolates **question selection and arithmetic**, not proof that a live judge would answer the probes correctly.

In [ ]:
narrow_metric, _ = await run_question_metric([Q_SHORT, Q_LONG])
extra_metric, _ = await run_question_metric(FIXED_QUESTIONS + [Q_NOT_IN_SOURCE])
variants = [
    ('Only two yield probes', narrow_metric),
    ('Add central price-risk probe', fixed_metric),
    ('Also add source-no fund probe', extra_metric),
    ('Generated-path five probes', generated_metric),
]
show_table(['Question set', 'Source-yes denominator', 'Coverage', 'Alignment', 'Overall'], [
    [label, sum(v.original_verdict == 'yes' for v in metric.coverage_verdicts),
     f"{metric.score_breakdown['Coverage']:.3f}", f"{metric.score_breakdown['Alignment']:.3f}", f'{metric.score:.3f}']
    for label, metric in variants
])
assert narrow_metric.score_breakdown['Coverage'] == 1
assert math.isclose(extra_metric.score_breakdown['Coverage'], 2 / 3)
assert extra_metric.coverage_verdicts[-1].original_verdict == 'no'

## Checks and interpretation

- The actual installed faithfulness metric calculated **2/3** from three authored verdicts.
- G-Eval normalized the authored raw **8** to **0.8**, using the repository's fixed rubric.
- `SummarizationMetric` calculated alignment **1.0**, question coverage **2/3**, and overall **2/3** for the fixed set.
- Removing the risk probe produced **1.0** question coverage while the risk warning remained absent. Omitted probes are blind spots.
- Omitting the list triggered one real question-generation step, answered offline with five authored questions. Its coverage was **3/5**.

Question sets make coverage inspectable, but their salience, wording, granularity, and source-answer quality matter. Equal-weight questions can overrepresent minor details. Our production workflow keeps faithfulness separate from holistic importance-based coverage, plus evidence-backed diagnostic findings; this experiment does not establish which judge is more accurate.

Try authoring another central probe, or an ambiguous/unanswerable one, and update `AUTHORED_ANSWERS` deliberately. Describe what should count as source evidence before changing responses. To evaluate actual generated judgments, use the live notebook and compare with human labels; don't reinterpret this scripted lab as model calibration.

## Next steps

Continue to the [practical workflow notebook](04_prompt_review_and_experiments.ipynb), or use [single-meeting evaluation](01_single_meeting.ipynb) for live judge calls and evidence diagnostics. Live calls incur provider usage; this lab has none.

Pinned implementation paths are inspectable above. Current primary references: [Faithfulness](https://deepeval.com/docs/metrics-faithfulness), [G-Eval](https://deepeval.com/docs/metrics-llm-evals), [Summarization](https://deepeval.com/docs/metrics-summarization). Recheck their behavior when upgrading DeepEval rather than assuming notebook arithmetic alone validates a new version.